# Milestone 3: fit and diagnostics

This notebook is generated by `tools/build_notebooks.py`. Edit the builder, not this file: a test fails if the two drift apart.

It follows course modules 7 and 8: fit the model, then ask whether what it leaves over is white noise. Three models are fitted on the training split: the AIC choice (the primary model, ADR-0002), BIC's choice, and the course's own recipe.

It reads committed files in `app/public/data/`: `fit.json` (estimates, roots, tests) and `residuals_aic.json` (the primary model's residual series). One cell near the end rebuilds them from `prices.json` and `aic_grid.json` (both committed) and checks they match; it skips if `prices.json` is absent.

The pictures are in the app (`cd app && npm run dev`): the **Fit and diagnostics** page.

In [ ]:
from arima_forecasting.artifacts import find_app_data, read_artifact

app_data = find_app_data()
fit = read_artifact(app_data, "fit")["data"]
models = {m["id"]: m for m in fit["models"]}
for m in fit["models"]:
    info = m["information"]
    print(f"{m['id']:>6}: ARIMA{tuple(m['order'])} trend {m['trend']!r}, {len(m['params'])} parameters, llf {info['llf']:.3f}, AIC {info['aic']:.3f}, BIC {info['bic']:.3f}, converged {m['converged']}")

## Module 7: the estimates, and what a p-value is

The course prints `model_fit.summary()`. The same numbers are in `fit.json`: each estimate, its standard error (statsmodels' default, the outer product of gradients), z = estimate / standard error, and the p-value. The p-value is the probability, if the true coefficient were 0, of a z at least as far from 0 as the one observed. It is not the probability that the coefficient is 0.

The two grid models are refitted from the estimates the grid recorded, so they are the grid's own points: the log-likelihood comes back within `llf_tolerance` of the grid's. That shows the point is reproduced, not that it is a maximum: another optimiser started there can still climb a little (the milestone-3 walkthrough measures it).

`sigma2` has a standard error but no z and no p-value: its null value 0 is on the boundary of what a variance can be, where z is not normal, so the test does not apply.

In [ ]:
for model_id in ("aic", "bic", "course"):
    m = models[model_id]
    gap = "n/a (not a grid cell)" if m["llf_gap"] is None else f"{m['llf_gap']:.2e}"
    print(f"{model_id}: llf minus the grid's: {gap}")
    for p in m["params"]:
        z = "    n/a" if p["z"] is None else f"{p['z']: .2f}"
        pv = "n/a" if p["p_value"] is None else f"{p['p_value']:.3g}"
        print(f"   {p['name']:>7} {p['value']: .6f}  se {p['std_err']:.6f}  z {z}  p {pv}")

## Roots that nearly cancel

An AR inverse root and an MA inverse root close together nearly divide out, and the model then behaves like one with two fewer parameters (concept note 09). The rule flags pairs whose two roots are within `cancel_distance` of each other in the complex plane: one to one, as many pairs as possible, and among those the least total distance.

In [ ]:
print("cancel_distance:", fit["cancel_distance"])
for model_id in ("aic", "bic", "course"):
    m = models[model_id]
    ar = [complex(r["re"], r["im"]) for r in m["inv_roots_ar"]]
    ma = [complex(r["re"], r["im"]) for r in m["inv_roots_ma"]]
    pairs = [(round(ar[p["ar"]].real, 3), round(ar[p["ar"]].imag, 3), round(p["distance"], 4)) for p in m["cancelling_pairs"]]
    print(f"{model_id}: AR moduli {[round(abs(r), 3) for r in ar]}, MA moduli {[round(abs(r), 3) for r in ma]}, cancelling pairs {len(pairs)}: {pairs}")

## The course's recipe, measured

The course runs `auto_arima` on the whole 2015-2023 series, test rows included, then refits `ARIMA(train, order=auto.order)` with every other argument at its default. Two things follow (ADR-0002, "What the course does"): the search saw the test rows, and the refit has no drift at d = 1 although the search chose one.

In [ ]:
t = fit["course_track"]
print("test rows the search saw:", t["test_rows_seen"], "of", t["n_course"])
print("full series chose:", t["full_series"]["order"], "intercept", t["full_series"]["with_intercept"], "| training rows alone chose:", t["train_only"]["order"], "intercept", t["train_only"]["with_intercept"], "| same:", t["same_order"])
dd = t["drift_dropped"]
print(f"refit trend {dd['refit_trend']!r}; with the drift put back: drift {dd['drift']:.6f} per day (p {dd['drift_p_value']:.3g}), AIC lower by {dd['aic_gap']:.3f}")

## Module 8: the residuals

The course plots `model_fit.resid` whole. Its first value is not a shock: with d = 1 the first one-step forecast has nothing to go on, so the first residual is about the log price itself. statsmodels leaves it out of the likelihood (`loglikelihood_burn`) and out of its own diagnostics; so does every test here.

Then three tests on the standardised residuals, each with its null hypothesis:

- **Ljung-Box** (no autocorrelation up to a lag), compared with a chi-squared on lag - (p + q); a lag at or below p + q cannot be tested.
- **Jarque-Bera** (skew 0 and kurtosis 3, as for a normal distribution).
- **ARCH-LM** (no volatility clustering: the squared residuals do not depend on their own past).

In [ ]:
for model_id in ("aic", "bic", "course"):
    dg = models[model_id]["diagnostics"]
    burned = dg["burned"][0]
    print(f"{model_id}: dropped {dg['burn_excluded']} residual(s): {burned['date']} resid {burned['resid']:.4f}; largest |resid| left {dg['max_abs_resid_after_burn']:.4f}")
    for r in dg["ljung_box"]:
        result = f"Q {r['statistic']:.3f}, df {r['df']}, p {r['p_value']:.3g}" if r["applied"] else f"Q {r['statistic']:.3f}, not applied (lag <= p + q = {r['model_df']})"
        print(f"   Ljung-Box lag {r['lag']:>2}: {result}")
    jb = dg["jarque_bera"]
    print(f"   Jarque-Bera {jb['statistic']:.1f}, p {jb['p_value']:.3g}, skew {jb['skew']:.3f}, kurtosis {jb['kurtosis']:.3f}")
    for r in dg["arch_lm"]:
        print(f"   ARCH-LM {r['lags']:>2} lags: LM {r['statistic']:.1f}, p {r['p_value']:.3g}")
    print("   verdicts:", dg["verdicts"])

## The prediction of concept note 07

Concept note 07 read the AIC choice's long lags as the grid fitting the volatility clusters in daily returns, and predicted that the residuals would cluster too. ARCH-LM tests that, and the ACF of the squared residuals shows it: under no clustering about alpha x nlags bars would pass the band by chance.

In [ ]:
cp = fit["clustering_prediction"]
print(f"clustering predicted for {cp['model_id']!r}: confirmed {cp['confirmed']}; ARCH-LM rejections {cp['arch_lm_rejections']}; squared-residual ACF past its band at {cp['acf_squared_outside_count']} of {fit['diagnostic_settings']['nlags']} lags (about {cp['acf_squared_expected_by_chance']} by chance)")
res = read_artifact(app_data, "residuals_aic")["data"]
print(f"residuals_aic: n {res['n']}, first date {res['rows'][0]['date']}, Student-t fit df {res['t_fit']['df']:.2f}")

## Rebuild the files from the committed inputs

`fit_payloads` takes the price table and the grid payload, refits the three models and recomputes every test. It runs the course's `auto_arima` too, so it takes a few seconds.

In [ ]:
from arima_forecasting.artifacts import table_from_prices
from arima_forecasting.export import fit_payloads

if not (app_data / "prices.json").is_file():
    print("prices.json is not in this copy - skipped")
else:
    table = table_from_prices(read_artifact(app_data, "prices")["data"])
    built = fit_payloads(table, read_artifact(app_data, "aic_grid")["data"])
    for name, payload in built.items():
        print(f"{name}.json matches a fresh computation:", payload == read_artifact(app_data, name)["data"])

## What comes next

Milestone 4 forecasts the test segment with these models, and scores them against the naive forecasts.